# GitHub Repository RAG

### Retrieval-Augmented Generation (RAG) system for querying and interacting with the contents of GitHub repositories.

---

**Author:** David Fernández Martínez  
**Email:** [david.fernxndez.martinez@gmail.com](mailto:david.fernxndez.martinez@gmail.com)  
**LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://linkedin.com/in/david-fernández-martínez)  
**GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## Build Evaluation Dataset for LLM-as-a-Judge Evaluation

This notebook builds an evaluation dataset to assess the overall quality of the RAG system using **Ragas and an LLM-as-a-judge approach**. This evaluation method uses an LLM to assess the quality of the generated responses based on the user query and the context retrieved by the system.

The dataset contains three key elements for each evaluation example:

* **Question:** the user's query submitted to the RAG system.
* **Contexts:** the document chunks retrieved to answer the query.
* **Answer:** the response generated by the LLM using the retrieved context.

The dataset was constructed using the [`spark-active-learning`](https://github.com/davidfernxndez/spark-active-learning) repository as the knowledge source. The RAG pipeline was configured with a **chunk size of 1,000 characters** and a **chunk overlap of 200 characters**.

### Notebook Objectives

* Define a representative set of 20 user queries covering different aspects of the repository.
* Run each query through the complete RAG pipeline.
* Collect the retrieved contexts and LLM-generated answers.
* Assemble and export the resulting dataset in `evaluation_dataset.json` file for subsequent evaluation with Ragas (see *notebooks/10_RAG_evaluation.ipynb*).

## Imports and Configuration

In [1]:
import json
import time
from pathlib import Path
from github_rag.ingestion.repository import GitHubRepositoryLoader
from github_rag.processing.processor import Processor
from github_rag.chunking.chunker import Chunker
from github_rag.embedding.embedding_model import create_embedding_model
from github_rag.vectordatabase.chroma_store import ChromaStore
from github_rag.retrieval.retriever import Retriever
from github_rag.prompt_engineering.prompt_builder import build_prompt
from github_rag.LLM_generation.generator import LLMGenerator

c:\Users\david\anaconda3\envs\github-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


To build the dataset, the same repository used for the demonstrations in the `notebooks/` directory is employed.

In [2]:
REPOSITORY_URL = "https://github.com/davidfernxndez/spark-active-learning"

## 1. Pipeline Setup

Run the complete RAG pipeline, ensuring that all the components are properly initialized and ready to use.

In [3]:
# Set the Loader Object with the URL
loader = GitHubRepositoryLoader(REPOSITORY_URL)

# Load the repository files as a list of LangChain Documents
documents = loader.load()

print(f"Number of documents ingested: {len(documents)}")

# Set the Processor Object
processor = Processor()

# Apply processing to the ingested documents
documents_processed = processor.process(documents)

print(f"Number of documents after processing: {len(documents_processed)}")

# Set the Chunker object that managed all the specific chunkers
chunker = Chunker()

# Apply chunking
documents_chunking = chunker.chunk(documents_processed)

print(f"Number of documents after chunking: {len(documents_chunking)}")

# Load the HuggingFace embedding model
start_time = time.time()
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = create_embedding_model(embedding_model_name)
print(f"Embedding model loaded time: {time.time() - start_time:.2f} seconds")

# Create the vector database storing
start_time = time.time()
vector_store = ChromaStore(
    mode="local",
    collection_name="github_repo"
)

# Create the database with documents and the embedding model
vector_store.create(
    documents=documents_chunking,
    embedding_model=embedding_model
)
print(f"Database generation time: {time.time() - start_time:.2f} seconds")

# Check the number of stored documents
stored_documents = vector_store.count()

print(f"Documents provided: {len(documents_chunking)}")
print(f"Documents stored:   {stored_documents}")

# Get the available files for the Retriever component
available_files = vector_store.get_unique_file_paths()
print(f"Number of available files: {len(available_files)}")

# Create the Retriever object
retriever = Retriever(vector_store)

# Set the LLM model
LLM_generator = LLMGenerator(mode="API")

Number of documents ingested: 12
Number of documents after processing: 45
Number of documents after chunking: 183
Embedding model loaded time: 7.44 seconds
Database generation time: 4.92 seconds
Documents provided: 183
Documents stored:   183
Number of available files: 12


## 2. Build the dataset

A total of 20 questions are defined to cover different aspects of the repository and a range of query complexities. The dataset is intentionally limited in size because both answer generation through the Groq API and the subsequent Ragas evaluation using the LLM-as-a-Judge approach consume API resources. Keeping the number of questions manageable helps control usage and avoid exceeding the API's daily limits.

In [4]:
queries =[
    "What clustering method is used for diversity?",
    "Why isn't the dataset included in the repository?",
    "What is the classifier trained and evaluated in the solution?",
    "How is uncertainty-based filtering performed?",
    "Which are the conclusions regarding the scalability of the solution?",
    "How is the instance closest to each cluster centroid selected?",
    "Which MLlib evaluator is used to define the metric in scripts/AL_methods.py?",
    "How many Active Learning iterations are executed in the scalability experiment of the script AL_scalability.py?",
    "How could the performance results of Active Learning be improved?",
    "How many samples were used for the 100%-sized reference dataset?",
    "¿What is the dataset used in the experiment",
    "How is the dataset preprocessed before training the model?",
    "How is data leakage prevented during feature standardization?",
    "How is the code of the developed solution structured?",
    "Where is broadcast performed in scripts/AL_methods.py?",
    "Which PySpark data structures are used?",
    "Explain the config variables defined in scripts/config.py",
    "Is the proposed solution scalable?",
    "Define the scalability measures used.",
    "What is the code used to initialize the labeling state?"
]

print(f"Number of queries defined {len(queries)}")

Number of queries defined 20


In [5]:
def format_evaluation_context(retrieval_chunks):
    """Format retrieved chunks as they are provided to the LLM."""
    contexts = []

    for chunk in retrieval_chunks:
        metadata = chunk.metadata

        contexts.append(
            f"File_name: {metadata['file_path']}\n"
            f"Source: {metadata['source']}\n"
            f"Content:\n{chunk.page_content}"
        )

    return contexts


# List to store the query, context and answer of each iteration
evaluation_dataset = []

for index, query in enumerate(queries, start=1):
    print(f"Query {index}/{len(queries)}")

    # Retrieve the relevant chunks
    retrieval_chunks = retriever.retrieve(query, available_files)

    # Build the prompt
    prompt = build_prompt(retrieval_chunks, query)

    # Query the LLM
    response = LLM_generator.generate(prompt)

    evaluation_dataset.append({
        "user_input": query,
        "retrieved_contexts": format_evaluation_context(retrieval_chunks),
        "response": response,
    })

    print("Results stored.")
    print("Sleeping 60 seconds...\n")
    # Sleep to avoid API limit requests
    time.sleep(60)

# Save evaluation dataset
output_path = Path("evaluation_dataset.json")

with output_path.open("w", encoding="utf-8") as file:
    json.dump(
        evaluation_dataset,
        file,
        ensure_ascii=False,
        indent=4,
    )

print(f"Evaluation dataset saved to: {output_path}")

Query 1/20
Results stored.
Sleeping 60 seconds...

Query 2/20
Results stored.
Sleeping 60 seconds...

Query 3/20
Results stored.
Sleeping 60 seconds...

Query 4/20
Results stored.
Sleeping 60 seconds...

Query 5/20
Results stored.
Sleeping 60 seconds...

Query 6/20
Results stored.
Sleeping 60 seconds...

Query 7/20
Results stored.
Sleeping 60 seconds...

Query 8/20
Results stored.
Sleeping 60 seconds...

Query 9/20
Results stored.
Sleeping 60 seconds...

Query 10/20
Results stored.
Sleeping 60 seconds...

Query 11/20
Results stored.
Sleeping 60 seconds...

Query 12/20
Results stored.
Sleeping 60 seconds...

Query 13/20
Results stored.
Sleeping 60 seconds...

Query 14/20
Results stored.
Sleeping 60 seconds...

Query 15/20
Results stored.
Sleeping 60 seconds...

Query 16/20
Results stored.
Sleeping 60 seconds...

Query 17/20
Results stored.
Sleeping 60 seconds...

Query 18/20
Results stored.
Sleeping 60 seconds...

Query 19/20
Results stored.
Sleeping 60 seconds...

Query 20/20
Results s